# ImmoScore: Sarah's house price notebook
A simple first model that predicts a house's sale price (`SalePrice`) from its numeric features. This is the "before MLOps" version from chapter 1.

## 1. Load the libraries

In [ ]:
import os
import pickle
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Load the data
Kaggle's `train.csv`, renamed `data.csv` and stored in `data/raw/`.

In [ ]:
df = pd.read_csv("../data/raw/data.csv")
df.shape

## 3. Keep the numeric columns and drop incomplete rows
`Id` is only a row number, not a feature of the house, so it goes too.

In [ ]:
df = df.select_dtypes("number").drop(columns="Id").dropna()
df.shape

## 4. Separate the features (X) from the target (y)

In [ ]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

## 5. Split into train and test, stratified on price bins
Prices are cut into 5 bins so train and test get the same mix of cheap and expensive houses.

The settings (5 bins, 20% test, seed 42) are written right here in the cell, the way Sarah did it. Atelier 2 moves them into `params.yaml`.

In [ ]:
bins = pd.qcut(y, q=5, labels=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=bins, random_state=42
)
len(X_train), len(X_test)

## 6. Train a linear regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

## 7. Measure the error on train and on test
A big gap between train and test means the model learned the training data by heart (overfitting).

In [ ]:
for name, X_part, y_part in [("train", X_train, y_train), ("test", X_test, y_test)]:
    pred = model.predict(X_part)
    rmse = mean_squared_error(y_part, pred) ** 0.5
    mae = mean_absolute_error(y_part, pred)
    r2 = r2_score(y_part, pred)
    print(f"{name:5}  RMSE {rmse:,.0f}  MAE {mae:,.0f}  R2 {r2:.3f}")

## 8. Save the model
It goes in `models/`, which `.gitignore` keeps off GitHub.

In [ ]:
os.makedirs("../models", exist_ok=True)
with open("../models/model.pkl", "wb") as f:
    pickle.dump(model, f)

## 9. Library versions
Write these down with your test score: different versions can give different numbers.

In [ ]:
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)

## What this notebook can't answer yet (chapter 1)
1. Which exact version of the data produced this score?
2. Will rerunning it in 6 months give the same score?
3. How does a website visitor get an estimate?